# Frozen MantisV2 baseline

This notebook uses the saved TelecomTS release and split IDs. It compares two frozen-encoder feature sets for fault detection and category prediction.

The current release has unresolved recording reviews. These results are provisional. The notebook uses the explicit exploratory option. It does not score the test split during model selection.

In [ ]:
from pathlib import Path
import os
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from clauseguard.baselines.artifacts import load_run
from clauseguard.baselines.mantis import run_mantis
from clauseguard.baselines.tree_data import split_data
from clauseguard.datasets.contracts import NUMERIC_CHANNELS
from clauseguard.datasets.scenarios import ScenarioDataset
from clauseguard.kpi.features import ChannelMedianFeatures

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "data/scenarios/release/release.json").exists())
os.chdir(ROOT)

RELEASE_DIR = Path("data/scenarios/release")
RF_REFERENCE = Path("artifacts/experiments/rf-all-data-provisional-2026-10-19")
XGBOOST_REFERENCE = Path("artifacts/experiments/xgboost-full-data-2026-10-19")
OUTPUT_DIR = Path("artifacts/experiments/mantis-notebook-demo")
if OUTPUT_DIR.exists():
    stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    OUTPUT_DIR = OUTPUT_DIR.with_name(f"{OUTPUT_DIR.name}-{stamp}")
CACHE_DIR = Path("artifacts/cache/mantis")
DATASET = ScenarioDataset(RELEASE_DIR, allow_exploratory=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DATASET.release_id, DATASET.splits["mode"], DEVICE)

## Inspect the input windows

Each window has 128 rows and 16 numeric KPI channels. The table shows the first five measurements from one training window. Labels and source details stay outside model inputs.

In [ ]:
train_cohort, train = split_data(DATASET, "detection", "train")
sample = ChannelMedianFeatures._window(train.inputs[0])[0]
display(pd.DataFrame(sample[:5], columns=NUMERIC_CHANNELS))
print({"windows": len(train.inputs), "shape": sample.shape, "target": train.targets[0]})

## Plot sample KPI windows

These plots help check how the synthetic fault windows differ from normal windows. The model still receives all 16 channels.

In [ ]:
members = DATASET.splits["membership"]
normal_id = next(sid for sid in train.scenario_ids if members[sid]["origin"] == "normal")
fault_id = next(sid for sid in train.scenario_ids if members[sid]["origin"] == "synthetic")
by_id = dict(zip(train.scenario_ids, train.inputs, strict=True))
fig, axes = plt.subplots(1, 3, figsize=(13, 3), sharex=True)
for axis, name in zip(axes, ("RSRP", "DL_BLER", "UL_SNR"), strict=True):
    index = NUMERIC_CHANNELS.index(name)
    for scenario_id, label in ((normal_id, "normal"), (fault_id, "synthetic fault")):
        values = ChannelMedianFeatures._window(by_id[scenario_id])[0][:, index]
        axis.plot(values, label=label)
    axis.set_title(name)
    axis.set_xlabel("Step")
axes[0].set_ylabel("KPI value")
axes[-1].legend()
fig.tight_layout()

## Train and select on development data

The runner fits medians, feature scaling, and logistic heads on train. It selects each task's feature set by development macro-F1. It checks RF and XGBoost references before fitting.

The command-line entry point uses the same runner: `uv run python -m clauseguard.cli.mantis train --release-dir data/scenarios/release --rf-reference artifacts/experiments/rf-all-data-provisional-2026-10-19 --xgboost-reference artifacts/experiments/xgboost-full-data-2026-10-19 --output-dir artifacts/experiments/mantis-baseline --allow-exploratory`.

In [ ]:
RESULT = run_mantis(
    DATASET, OUTPUT_DIR, rf_reference=RF_REFERENCE,
    xgboost_reference=XGBOOST_REFERENCE, cache_dir=CACHE_DIR,
    allow_exploratory=True, device=DEVICE,
)
pd.DataFrame([
    {"task": task, "selected features": row["selected_feature_variant"],
     "dev macro-F1": row["dev_macro_f1"], "train windows": row["train_windows"],
     "dev windows": row["dev_windows"]}
    for task, row in RESULT["tasks"].items()
])

## Compare development runs

The table compares Mantis with RF and XGBoost on matching cases. Read per-class scores and prediction failures with macro-F1.

In [ ]:
comparison_rows = []
for task, row in RESULT["tasks"].items():
    for model, result in row["comparison"]["runs"].items():
        comparison_rows.append({"task": task, "model": model,
                              "macro-F1": result["metrics"]["macro_f1"],
                              "fault recall": result["metrics"].get("recall"),
                              "failed predictions": result["prediction_status_counts"].get("error", 0)})
display(pd.DataFrame(comparison_rows))

## Review errors by fault origin

This table reports development errors by source origin. It keeps the full cohort visible. It does not use test labels or change the selected feature set.

In [ ]:
error_rows = []
for task, row in RESULT["tasks"].items():
    run = load_run(OUTPUT_DIR / row["selected_run"])
    expected = run["expected"]
    for prediction in run["predictions"]:
        origin = members[prediction.scenario_id]["origin"]
        error_rows.append({"task": task, "origin": origin,
                           "expected": expected[prediction.scenario_id],
                           "predicted": prediction.predicted_label,
                           "status": prediction.status,
                           "error": prediction.error})
errors = pd.DataFrame(error_rows)
errors["wrong"] = errors["expected"] != errors["predicted"]
display(errors.groupby(["task", "origin"], dropna=False).agg(
    windows=("expected", "size"), errors=("wrong", "sum"),
    failed=("status", lambda values: int((values != "ok").sum())),
).reset_index())

## Replay a saved run

Use the CLI `predict` command to reload a saved head. Keep test scoring separate from training and selection. The saved run records the encoder revision, weight checksum, preprocessing values, class weights, and feature cache identity.